# 第7周 · 神经元、计算图与 micrograd

**本周结束时你能做到**：解释神经元 = 加权和 + 非线性，以及为什么没有非线性就解不了 XOR；在纸上对一个小计算图做前向与反向；从零实现一个标量自动微分引擎，并用数值梯度检查和 PyTorch 对照验证它；用它搭出 `Neuron → Layer → MLP`，在月牙数据上训练出弯曲的决策边界。

本周的实现蓝本是 Andrej Karpathy 的 [micrograd](https://github.com/karpathy/micrograd)（MIT 许可）与其配套视频 [The spelled-out intro to neural networks and backpropagation](https://www.youtube.com/watch?v=VMj-3S1tku0)。我们照视频用 **tanh** 做主角（仓库版只有 relu），作业里你会把整套运算补全。

In [ ]:
# 运行这一格来安装课程工具包（每次在 Colab 打开 notebook 后运行一次）
import importlib.util, subprocess, sys
if importlib.util.find_spec("zero2researcher") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/ReaganGen/Zero2Researcher"], check=True)
print("课程工具包已就绪")

In [ ]:
import math
import random

import numpy as np
import matplotlib.pyplot as plt
from zero2researcher.checks import week07 as ck

---
## S1 · 从逻辑回归到神经元

第5周的逻辑回归：输入加权求和，过一个 sigmoid。一个**神经元**几乎是同一回事：

$$\text{out} = \varphi\Big(\sum_i w_i x_i + b\Big)$$

加权和（线性部分）+ 一个**非线性激活函数** $\varphi$（本周用 $\tanh$，输出压在 $-1$ 到 $1$ 之间）。1958 年的**感知机**就是"加权和 + 阶跃函数"——风光了十年，直到有人指出：它解不了下面这个只有 4 个点的小问题。

### XOR：线性模型的滑铁卢

XOR（异或）：两个输入**不同**时输出 1，相同时输出 0。拿第5周的逻辑回归试试：

In [ ]:
from sklearn.linear_model import LogisticRegression

X_xor = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
y_xor = np.array([0, 1, 1, 0])

lr_model = LogisticRegression().fit(X_xor, y_xor)
print("逻辑回归在 XOR 上的训练准确率:", lr_model.score(X_xor, y_xor))

# 画出 4 个点和决策区域：一条直线无论怎么摆，最多只能"猜对一半"
xx, yy = np.meshgrid(np.linspace(-0.5, 1.5, 200), np.linspace(-0.5, 1.5, 200))
Z = lr_model.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
fig, ax = plt.subplots(figsize=(4, 4))
ax.contourf(xx, yy, Z, alpha=0.2, cmap="coolwarm")
ax.scatter(X_xor[:, 0], X_xor[:, 1], c=y_xor, cmap="coolwarm", s=120, edgecolors="k")
ax.set_xlabel("x1")
ax.set_ylabel("x2")
ax.set_title("No single line separates XOR")
plt.show()

### 为什么需要非线性

"多叠几层线性层"救不了感知机：两个线性层叠起来还是一个线性层（矩阵乘法可以先结合）。小实验证明：

In [ ]:
rng = np.random.default_rng(42)
W1 = rng.normal(size=(2, 4))      # 第一层：2 维 → 4 维
W2 = rng.normal(size=(4, 1))      # 第二层：4 维 → 1 维
x = rng.normal(size=(5, 2))       # 5 个输入点

two_layers = x @ W1 @ W2
one_layer = x @ (W1 @ W2)         # 两个矩阵先乘成一个 2→1 的矩阵
print("两层线性 与 一层线性 的最大差异:", np.abs(two_layers - one_layer).max())

差异是 0：不加非线性，一百层也只是一条直线。而只要在中间**夹一个 tanh**，两个神经元就够解 XOR——下面的权重是我手工设好的：

In [ ]:
def tiny_mlp(x1, x2):
    """手工设好权重的 2-2-1 网络：隐层算出 OR 与 AND，输出层组合成 XOR"""
    h_or = np.tanh(10 * (x1 + x2 - 0.5))     # 只有 (0,0) → -1，其余 → +1
    h_and = np.tanh(10 * (x1 + x2 - 1.5))    # 只有 (1,1) → +1
    out = np.tanh(10 * (h_or - h_and - 0.5))
    return int(out > 0)

for (x1, x2), y in zip(X_xor, y_xor):
    print(f"({x1}, {x2}) → 预测 {tiny_mlp(x1, x2)}   正确答案 {y}")

隐层把输入**重新表示**成两个新特征（OR 与 AND）——在新表示里，XOR 变成了线性可分问题。这就是"深度"的全部秘密：每一层学一种更好用的表示。本周剩下的问题只有一个：这些权重刚才是我手设的，**机器怎么自己学出来？** 答案 = 梯度下降（第4周）+ 自动算梯度（今天的主菜）。

### 动手试：TensorFlow Playground

打开 [https://playground.tensorflow.org](https://playground.tensorflow.org)（纯浏览器运行，无需安装），跟着做：

1. 数据选 **XOR**（左侧第三个图案），激活函数（Activation）选 **Tanh**，把隐层删到 **0 层**——点运行，损失卡住，边界是一条直线（刚才的滑铁卢）；
2. 加回 **1 个隐层、4 个神经元**——重新运行，看边界弯出"棋盘格"；
3. 把数据换成**螺旋**，试试需要几层、每层几个神经元才能学好；
4. 把激活函数换成 **Linear**——再多的层也学不动，非线性的必要性眼见为实。

---
## S2 · 计算图：先在纸上做一遍反向传播

任何表达式都能画成**计算图**。以 $L = (a \cdot b + c)^2$ 为例，取 $a=2,\ b=-3,\ c=10$：

```
a=2 ──┐
      (×)── e=-6 ──┐
b=-3 ─┘            (+)── d=4 ──(平方)── L=16
c=10 ──────────────┘
```

**前向**：从左到右算出每个中间值。**反向**：从 $L$ 出发，从右到左问每个节点："你动一点点，$L$ 动多少？"——这个数叫该节点的**梯度**（代码里记作 `grad`）。

| 节点 | 前向值 | 梯度 $\partial L/\partial(\cdot)$ | 怎么来的 |
| --- | --- | --- | --- |
| $L$ | $16$ | $1$ | $L$ 对自己的导数 |
| $d$ | $4$ | $2d = 8$ | 平方的导数 $2d$ |
| $c$ | $10$ | $8 \times 1 = 8$ | 加法：原样放行 |
| $e$ | $-6$ | $8 \times 1 = 8$ | 加法：原样放行 |
| $a$ | $2$ | $8 \times b = -24$ | 乘法：乘上另一个因子 |
| $b$ | $-3$ | $8 \times a = 16$ | 乘法：乘上另一个因子 |

规律（**链式法则**）：每个节点的梯度 = **下游传来的梯度 × 本步运算的局部导数**。加法的局部导数是 1（原样放行），乘法的局部导数是另一个因子的值。

不信可以"亲手拨一下"验证——把 $a$ 挪动一小步，看 $L$ 动了多少倍：

In [ ]:
a, b, c = 2.0, -3.0, 10.0
L = (a * b + c) ** 2
h = 1e-4
L_nudged = ((a + h) * b + c) ** 2
print("dL/da ≈", (L_nudged - L) / h, "  （纸上算的是 -24）")

### 动手写：让 Python 记住计算过程

要把"反向走一遍"写成程序，先得让每个数**记住自己是怎么算出来的**。这就是 micrograd 的 `Value` 类（第一版）：`data` 存数值，`_prev` 存"我由谁算出来"，`_op` 存"用的什么运算"。写了 `__add__` 和 `__mul__` 之后，`+` 和 `*` 就会自动把表达式连成一张计算图。

In [ ]:
class Value:
    """第一版：只记录前向。data 是数值，_prev 是产生它的输入，_op 是运算名"""

    def __init__(self, data, _children=(), _op=""):
        self.data = data
        self._prev = set(_children)
        self._op = _op

    def __repr__(self):
        return f"Value(data={self.data})"

    def __add__(self, other):          # 写 a + b 时 Python 自动调用 a.__add__(b)
        # 在这里写你的代码
        raise NotImplementedError

    def __mul__(self, other):
        # 在这里写你的代码
        raise NotImplementedError


a, b, c = Value(2.0), Value(-3.0), Value(10.0)
d = a * b + c
L = d * d              # 还没有乘方运算，平方先写成自己乘自己
print(L)

In [ ]:
def print_graph(v, indent=""):
    """用缩进打印计算图（文本足够看清结构，不引入 graphviz）"""
    print(f"{indent}{v._op or '叶子'}  data={v.data}")
    for child in v._prev:
        print_graph(child, indent + "    ")

print_graph(L)

In [ ]:
# 顺序反过来就坏了：写 1.0 + a 时 Python 先问 float "你会加 Value 吗"——它不会，
# 我们也还没告诉 Python 怎么办。读一读这个报错，第二版里我们用 __radd__ 修复它
1.0 + a

---
## S3 · backward：让梯度自己流回去

第二版 `Value` 新增三样东西：

1. **`grad`**：存"最终输出对我的导数"，初始为 0；
2. **`_backward` 闭包**：每个运算在产生输出节点时，顺手把"怎么把梯度传回输入"写成一个小函数挂在输出上。链式法则在代码里就一行：`self.grad += out.grad × 局部导数`；
3. **`backward()`**：对整张图做**拓扑排序**（保证先处理下游、再处理上游），把自己的 `grad` 设为 1，然后按反序逐个调用 `_backward`。

两个必须说透的细节：

- **为什么是 `+=` 而不是 `=`**：同一个节点可能出现在多条路径上（比如 `b = a + a`），每条路径都会传回一份梯度，必须**累加**。写 `=` 会让后算的路径覆盖先算的——micrograd 视频在 1:22:28 专门修这个 bug，它也是本周作业里最常见的错误。
- **tanh 的局部导数**：$\tanh'(x) = 1 - \tanh^2(x)$。前向已经算出了 $t=\tanh(x)$，反向直接用 $1 - t^2$，一次都不用重算。

In [ ]:
class Value:
    """第二版：完整的标量自动微分节点（作业里你会补全更多运算）"""

    def __init__(self, data, _children=(), _op=""):
        self.data = data
        self.grad = 0.0                   # ∂输出/∂self，初始为 0
        self._backward = lambda: None     # 叶子节点没有输入，反向什么都不用做
        self._prev = set(_children)
        self._op = _op

    def __repr__(self):
        return f"Value(data={self.data}, grad={self.grad})"

    def __add__(self, other):
        # 在这里写你的代码
        raise NotImplementedError

    def __mul__(self, other):
        # 在这里写你的代码
        raise NotImplementedError

    def tanh(self):
        # 在这里写你的代码
        raise NotImplementedError

    def relu(self):                   # S4 的 hinge 损失要用 max(0, x)，这里先给出
        out = Value(0.0 if self.data < 0 else self.data, (self,), "ReLU")

        def _backward():              # 正半边放行梯度，负半边拦下
            self.grad += (out.data > 0) * out.grad
        out._backward = _backward
        return out

    # 两个"搭便车"运算：让 1.0 + v 和 2 * v 也能用（作业里补全整套）
    def __radd__(self, other):
        return self + other

    def __rmul__(self, other):
        return self * other

    def backward(self):
        # 在这里写你的代码
        raise NotImplementedError


In [ ]:
# 在纸上那个例子上验收：d 被 L = d * d 用了两次，正好考验 "+="
a, b, c = Value(2.0), Value(-3.0), Value(10.0)
d = a * b + c
L = d * d
L.backward()
print(f"a.grad = {a.grad}   b.grad = {b.grad}   c.grad = {c.grad}   （纸上：-24, 16, 8）")

ck.value_add(Value)
ck.value_mul(Value)
ck.value_tanh(Value)
ck.gradient_reuse(Value)

### 梯度检查：给自动微分做"阳性对照"

我们刚写的 `backward` 对不对？第6周的习惯立刻用上：**数值差分**不依赖我们的任何代码，拿它当阳性对照。CS231n 的标准做法：

- **中心差分** $\dfrac{f(x+h) - f(x-h)}{2h}$，$h = 10^{-5}$；
- 比较用**相对误差** $\dfrac{|a - b|}{\max(|a|, |b|)}$：$<10^{-7}$ 很好；$\sim 10^{-4}$ 只对带 kink 的函数（如 relu 在 0 附近）可接受；$>10^{-2}$ 基本错了。

In [ ]:
def numerical_grad(f, x, h=1e-5):
    """中心差分估计 f 在 x 处的导数"""
    # 在这里写你的代码
    raise NotImplementedError


def rel_error(g1, g2):
    return abs(g1 - g2) / max(abs(g1), abs(g2), 1e-12)


# 同一个函数 f(x) = tanh(2x + 1)：解析梯度走我们的 backward，数值梯度走差分
x = Value(0.5)
y = (2 * x + 1).tanh()
y.backward()
numeric = numerical_grad(lambda t: math.tanh(2 * t + 1), 0.5)

print(f"解析梯度 {x.grad:.10f}   数值梯度 {numeric:.10f}   相对误差 {rel_error(x.grad, numeric):.2e}")
ck.numerical_grad(numerical_grad)

### 第二道对照：PyTorch 的 autograd

PyTorch 内部做的事和我们这 60 行**本质相同**（只是节点从标量换成张量、实现换成 C++）。同一个表达式，两边的梯度应当一字不差——torch 侧用 `dtype=torch.float64`，和 Python 小数同精度：

In [ ]:
import torch

# micrograd 这边
x, w = Value(-0.6), Value(1.8)
out = (x * w + x).tanh() * w
out.backward()

# PyTorch 这边：同一个表达式
# 在这里写你的代码
raise NotImplementedError

rel = max(rel_error(x.grad, tx.grad.item()), rel_error(w.grad, tw.grad.item()))
print(f"x.grad: 我们 {x.grad:.12f}   torch {tx.grad.item():.12f}")
print(f"w.grad: 我们 {w.grad:.12f}   torch {tw.grad.item():.12f}")
ck.torch_agreement(rel)

---
## S4 · Neuron → Layer → MLP：搭网络、训网络

照 micrograd 的 `nn.py` 搭三个类（第3周的"类"知识全用上）：

- **`Neuron`**：S1 的公式——权重随机 $U(-1,1)$、偏置 0、加权和过 tanh；输出层不要非线性（`nonlin=False`）；
- **`Layer`**：一排各自独立的 Neuron；
- **`MLP`**：一串 Layer，最后一层自动线性输出。

基类 `Module` 管两件杂事：收集全部参数（`parameters`），训练前清零梯度（`zero_grad`）。**忘了清零**，上一步的梯度会累加进这一步——这是 `+=` 设计的另一面，记住口诀"每步先清零"。

In [ ]:
class Module:
    def zero_grad(self):
        for p in self.parameters():
            p.grad = 0.0

    def parameters(self):
        return []


class Neuron(Module):
    def __init__(self, nin, nonlin=True):
        self.w = [Value(random.uniform(-1, 1)) for _ in range(nin)]
        self.b = Value(0.0)
        self.nonlin = nonlin

    def __call__(self, x):            # n(x) 这样调用时执行；x 是 Value 的列表
        # 在这里写你的代码
        raise NotImplementedError

    def parameters(self):
        return self.w + [self.b]


class Layer(Module):
    def __init__(self, nin, nout, **kwargs):
        self.neurons = [Neuron(nin, **kwargs) for _ in range(nout)]

    def __call__(self, x):
        out = [n(x) for n in self.neurons]
        return out[0] if len(out) == 1 else out    # 只有 1 个输出时直接给标量

    def parameters(self):
        return [p for n in self.neurons for p in n.parameters()]


class MLP(Module):
    def __init__(self, nin, nouts):               # MLP(2, [8, 8, 1])：2 维输入、三层
        sz = [nin] + nouts
        self.layers = [Layer(sz[i], sz[i + 1], nonlin=(i != len(nouts) - 1))
                       for i in range(len(nouts))]

    def __call__(self, x):
        # 在这里写你的代码
        raise NotImplementedError

    def parameters(self):
        return [p for layer in self.layers for p in layer.parameters()]


random.seed(42)
model = MLP(2, [8, 8, 1])
print("参数个数:", len(model.parameters()))
print("随机初始权重下对点 (1.0, -1.0) 的打分:", model([Value(1.0), Value(-1.0)]))

### 月牙数据与 max-margin 损失

训练数据用 `make_moons`：两弯月牙互相咬合，直线分不开——正好逼出非线性。损失、正则与学习率衰减照 micrograd 的 demo 设定：

- 标签从 0/1 变成 **±1**；网络输出一个打分 $s$：$s>0$ 判正类、$s<0$ 判负类；
- 损失用 **max-margin（hinge）**：$\ell_i = \max(0,\ 1 - y_i s_i)$——打分方向对**且**离 0 足够远（间隔 ≥ 1）才算无罪，方向错罚得更重；
- 外加一点 **L2 正则** $\alpha \sum_p p^2$（$\alpha = 10^{-4}$），不让权重乱长。

（第5周见过的交叉熵在这里也完全可用；本课跟随蓝本用 hinge，第8周起我们统一用交叉熵。）

**和 demo 不一样的三处**（对照原 notebook 时别困惑）：① 网络从 demo 的 `MLP(2, [16, 16, 1])` 缩到 `MLP(2, [8, 8, 1])`——纯 Python 逐标量训练很慢，16-16 宽的网络 100 步要一分钟上下（机器忙时更久），等得太久；② 样本数用 150 个（demo 是 100 个）；③ 激活函数用 S2 写的 tanh（micrograd 仓库版用 relu）。作业 C 会回到 demo 的 100 个点，专门比较宽度 4 与 16。

In [ ]:
from sklearn.datasets import make_moons

X_moons, y01 = make_moons(n_samples=150, noise=0.1, random_state=42)
y_moons = 2 * y01 - 1          # 0/1 → -1/+1

fig, ax = plt.subplots(figsize=(5, 4))
ax.scatter(X_moons[:, 0], X_moons[:, 1], c=y01, cmap="coolwarm", s=18)
ax.set_xlabel("x1")
ax.set_ylabel("x2")
ax.set_title("make_moons: no straight line separates the two arcs")
plt.show()

In [ ]:
def loss_fn(model, X, y, alpha=1e-4):
    """全批量 hinge 损失 + L2 正则；顺带返回当前训练准确率"""
    # 在这里写你的代码
    raise NotImplementedError
    acc = sum((s.data > 0) == (yi > 0) for s, yi in zip(scores, y)) / len(y)
    return total, acc

### 训练循环：第4周的梯度下降，梯度改为自动

每一步四件事，顺序背下来（第8周的 PyTorch 循环一模一样）：

**① 前向算损失 → ② `zero_grad()` 清零 → ③ `backward()` 算梯度 → ④ 沿负梯度更新。**

学习率从 1.0 线性衰减到 0.1：开头大步快走，后期小步精修。

> 纯 Python 逐标量建图、传梯度，150 个样本每步约 0.25 秒、100 步约半分钟——先体会"每个标量的梯度都被亲手算了一遍"，第8周 PyTorch 把同样的事做到百万参数上。

In [ ]:
# 决策边界的取样网格（步长 0.25，照 micrograd demo），训练途中拍三张快照
gx = np.arange(X_moons[:, 0].min() - 0.5, X_moons[:, 0].max() + 0.5, 0.25)
gy = np.arange(X_moons[:, 1].min() - 0.5, X_moons[:, 1].max() + 0.5, 0.25)
GX, GY = np.meshgrid(gx, gy)
grid = [[(float(x1), float(x2)) for x1, x2 in zip(rx, ry)] for rx, ry in zip(GX, GY)]

In [ ]:
random.seed(42)
model = MLP(2, [8, 8, 1])

history, snapshots = [], {}
for k in range(100):
    # 在这里写你的代码
    raise NotImplementedError
    history.append((loss.data, acc))
    if k in (0, 10, 99):                           # 拍决策边界快照
        snapshots[k] = [[model([Value(x1), Value(x2)]).data for x1, x2 in row] for row in grid]
    if k % 10 == 0 or k == 99:
        print(f"step {k:3d}   loss {loss.data:.4f}   acc {acc:.3f}")

train_acc = history[-1][1]
ck.mlp_accuracy(train_acc)

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(14, 3.2))
axes[0].plot([h[0] for h in history])
axes[0].set_xlabel("Step")
axes[0].set_ylabel("Hinge loss")
axes[0].set_title("Training loss")
for ax, (k, Z) in zip(axes[1:], sorted(snapshots.items())):
    ax.contourf(GX, GY, np.array(Z) > 0, alpha=0.25, cmap="coolwarm")
    ax.scatter(X_moons[:, 0], X_moons[:, 1], c=y01, cmap="coolwarm", s=10)
    ax.set_xlabel("x1")
    ax.set_ylabel("x2")
    ax.set_title(f"Decision boundary after step {k + 1}")
plt.tight_layout()
plt.show()

### 本周小结

- 神经元 = 加权和 + 非线性；没有非线性，再深也只是一条直线（XOR 滑铁卢）
- 计算图：前向存 `data`，反向传 `grad`；链式法则在代码里就一行 `self.grad += out.grad × 局部导数`
- 梯度累加必须 `+=`：同一节点多条路径，各传一份；每步训练前必须 `zero_grad()`
- `backward()` = 拓扑排序 + 逐节点 `_backward`；正确性靠两道对照——数值差分（中心差分 + 相对误差）与 PyTorch autograd
- Neuron → Layer → MLP 共约 50 行；hinge + L2、学习率衰减的 SGD，150 个点 100 步训练准确率 100%，决策边界从直线弯成月牙

**预告**：第8周的 PyTorch **autograd 就是张量版的 micrograd**——`loss.backward()` 那一行背后，正是你今天写的拓扑排序和 `_backward` 闭包，只是每个节点从一个标量换成了一整块张量。